# 人工知能 第3回 演習課題

## 2層ニューラルネットワークの実装

1. 下記要件を満たす2層ニューラルネットワークの重みとバイアスの初期値を生成し、それらを辞書（dict）に格納したものを返す**関数**を実装せよ。
   - 関数名は`setup_two_layer_network`とする
   - 引数はなし
   - 戻り値として重みとバイアスを管理するリストを返す
   - 入力する要素数は5（1×5）
   - 出力する要素数は3（1×3）
   - 重み・バイアスは乱数から生成する
   - 各層のノード数は第1層が4、第2層が3

2. 下記要件を満たす2層ニューラルネットワークに入力$X$を渡し、その出力を戻り値として返す**関数**を実装せよ。
   - 関数名は`forward_two_layer_network`とする
   - 引数として入力`x`を受け取る
   - 1層目の活性化関数はReLU関数
   - 2層目の活性化関数はソフトマックス関数

3. 問1・問2で実装したコードを任意の生成AIに渡し、どのような処理を行うコードか、どんな理論に基づいて実装されたコードであるか、より良い実装とするにはどこに手を加えればよいかを解説させよ。その後、以下内容を本問題の回答として記載せよ。
   - 利用した生成AIに関する情報（サービス名・モデル名・バージョン・URLなど）
   - 生成AIに渡したプロンプト**全文**
   - 生成AIが出力した回答内容**全文**

4. 問3の内容を踏まえ、問1・問2のコードを改修せよ。

**利用するモジュールのインポート**

In [1]:
import numpy as np

**利用するメソッドの実装**

In [2]:
def relu(x):
    return np.maximum(0, x)

def softmax(x):
    c = np.max(x)
    return np.exp(x-c) / np.sum(np.exp(x-c))

In [3]:
# 問1 回答欄
def setup_two_layer_network():
    # 入力する要素数は5（1×5）、第1層のノード数は4
    # W1の形状: (5, 4)
    W1 = np.random.randn(5, 4)
    b1 = np.random.randn(4)
    
    # 第1層のノード数は4、出力する要素数は3（1×3 / 第2層のノード数）
    # W2の形状: (4, 3)
    W2 = np.random.randn(4, 3)
    b2 = np.random.randn(3)
    
    network = {
        'W1': W1,
        'b1': b1,
        'W2': W2,
        'b2': b2
    }
    return network

In [4]:
# 問2 回答欄
def forward_two_layer_network(network, x):
    W1, b1 = network['W1'], network['b1']
    W2, b2 = network['W2'], network['b2']
    
    # 1層目の計算（活性化関数は ReLU）
    a1 = np.dot(x, W1) + b1
    z1 = relu(a1)
    
    # 2層目の計算（活性化関数はソフトマックス関数）
    a2 = np.dot(z1, W2) + b2
    y = softmax(a2)
    
    return y

**問3 回答欄**


サービス名：GoogleGemini
モデル名：Gemini 1.5 Pro
バージョン：最新版

〈プロンプト〉
下記要件を満たす2層ニューラルネットワークの初期値生成関数 setup_two_layer_network と、フォワード処理を行う関数 forward_two_layer_network のコードを提示し、その処理内容や理論、より良い実装にするための改善点を解説してください。

〈回答〉
ご提示いただいたコードおよび要件に基づき、以下に処理内容と理論、改善点を解説します。処理内容と理論的背景重み・バイアスの初期化: 各層の結合重み $W$ はランダムな正規分布に従う乱数で初期化され、バイアス $b$ も同様に初期値が与えられます。これにより、対称性の破壊が行われ、学習が正常に進むようになります。順伝播（Forward Propagation）: 入力データ $x$ と重みの線形結合にバイアスを加え、1層目では非線形性を付与するために ReLU 関数を適用しています。さらに2層目では、出力の総和が1となる確率分布を得るためにソフトマックス関数を使用しています。より良い実装にするための改善点重みの初期値のスケール調整: np.random.randn をそのまま使用すると、ノード数が増えた際に分散が大きくなり（または小さくなり）、勾配消失・爆発の原因になります。一般的には Heの初期値（He initialization）や Xavierの初期値など、前後の層のノード数に応じたスケーリング（例：np.random.randn(...) * np.sqrt(2.0 / input_size)）を導入することが推奨されます。バッチ処理への対応: 現在の1サンプルを想定した実装から、複数データを同時に処理するミニバッチ学習に対応させると、行列計算の効率（ベクトル化）が向上します。

In [5]:
# 問4 回答欄
def setup_two_layer_network():
    # Heの初期値（ReLUに適したスケーリング）を適用した重みの初期化
    W1 = np.random.randn(5, 4) * np.sqrt(2.0 / 5)
    b1 = np.zeros(4)  # バイアスは一般的に0から開始することも多い
    
    W2 = np.random.randn(4, 3) * np.sqrt(2.0 / 4)
    b2 = np.zeros(3)
    
    network = {
        'W1': W1,
        'b1': b1,
        'W2': W2,
        'b2': b2
    }
    return network

def forward_two_layer_network(network, x):
    W1, b1 = network['W1'], network['b1']
    W2, b2 = network['W2'], network['b2']
    
    # 1層目（ReLU）
    a1 = np.dot(x, W1) + b1
    z1 = relu(a1)
    
    # 2層目（ソフトマックス）
    a2 = np.dot(z1, W2) + b2
    y = softmax(a2)
    
    return y